# 01 — Data Loading and Initial Inspection

This notebook loads the **AI4I 2020 Predictive Maintenance Dataset** from the repository when available and otherwise downloads it directly from the UCI Machine Learning Repository. This makes the notebook runnable in a fresh Google Colab session without manually uploading data.

The initial inspection covers dataset dimensions, data types, missing values, duplicate observations, feature cardinality, descriptive statistics, and target-class distribution.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


In [2]:
from pathlib import Path

UCI_DATA_URL = "https://archive.ics.uci.edu/static/public/601/ai4i%2B2020%2Bpredictive%2Bmaintenance%2Bdataset.zip"

# Resolve the repository root when running locally. In a fresh Colab runtime,
# no repository data folder exists, so the notebook falls back to UCI.
def find_repo_root():
    cwd = Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    return cwd

REPO_ROOT = find_repo_root()
LOCAL_DATA_PATH = REPO_ROOT / "data" / "raw" / "ai4i2020.csv"

if LOCAL_DATA_PATH.exists():
    df = pd.read_csv(LOCAL_DATA_PATH)
else:
    df = pd.read_csv(UCI_DATA_URL, compression="zip")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")


Rows: 10000
Columns: 14


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [4]:
df.isna().sum()

,0
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [7]:
df.columns.tolist()

['UDI',
 'Product ID',
 'Type',
 'Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]',
 'Machine failure',
 'TWF',
 'HDF',
 'PWF',
 'OSF',
 'RNF']

In [8]:
df.nunique()

,0
UDI,10000
Product ID,10000
Type,3
Air temperature [K],93
Process temperature [K],82
Rotational speed [rpm],941
Torque [Nm],577
Tool wear [min],246
Machine failure,2
TWF,2


In [9]:
df["Machine failure"].value_counts()

,count
Machine failure,
0,9661
1,339


In [10]:
df["Machine failure"].value_counts(normalize=True).mul(100).round(2)

,proportion
Machine failure,
0,96.61
1,3.39


## Target Class Distribution

The target variable is strongly imbalanced: 9,661 observations (96.61%) represent no machine failure, while only 339 observations (3.39%) represent machine failure.

This imbalance will be important during modelling. A classifier that predicts "no failure" for every observation would achieve approximately 96.6% accuracy while detecting none of the actual failures. Therefore, model evaluation should not rely on accuracy alone. Metrics such as precision, recall, F1-score and ROC-AUC will also be considered.

## Descriptive Statistics


In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
UDI,10000.0,5000.50000,2886.895680,1.0,2500.75,5000.5,7500.25,10000.0
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.30,300.1,301.50,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.80,310.1,311.10,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.00,1503.0,1612.00,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.20,40.1,46.80,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.00,108.0,162.00,253.0
Machine failure,10000.0,0.03390,0.180981,0.0,0.00,0.0,0.00,1.0
TWF,10000.0,0.00460,0.067671,0.0,0.00,0.0,0.00,1.0
HDF,10000.0,0.01150,0.106625,0.0,0.00,0.0,0.00,1.0
PWF,10000.0,0.00950,0.097009,0.0,0.00,0.0,0.00,1.0


## Save Processed Dataset

The inspected dataset is also written to Parquet as a compact processed artifact. Notebooks 02 and 03 remain independently runnable and therefore do not require this file to be created in the same Colab session.


In [12]:
OUTPUT_PATH = REPO_ROOT / "data" / "processed" / "ai4i2020.parquet"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(OUTPUT_PATH, index=False)

df_check = pd.read_parquet(OUTPUT_PATH)

print(df_check.shape)


(10000, 14)
